# Phase 1: PDF Extraction & Image Saving

This notebook discovers PDFs, extracts structured text using Docling, saves extracted images to disk, and caches the structured data as JSON.
This separates the extraction from the LLM generation so timeouts don't force you to re-parse the PDFs.

In [1]:
CONFIG = {
    "pdf_dataset_root": "/kaggle/input/textbooks",
    "output_root": "/kaggle/working/idp_curriculum_generation",
    "include_subjects": [],
    "include_grades": [],
    "include_pdf_name_contains": "",
    "max_pdfs": None,
    "max_sections": None,
    "debug_discovery": True,
    "docling_do_ocr": False,
    "docling_do_table_structure": True,
    "docling_generate_picture_images": True,  # ENABLED IMAGE EXTRACTION
    "docling_generate_page_images": False,
    "min_section_words": 500,
    "max_section_words": 1500,
    "resume": False,
}


In [2]:
!pip -q install docling pandas pillow opencv-python tqdm

import importlib, json, logging, os, re, sys, time, hashlib, zipfile
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("idp-curriculum-extraction")

OUTPUT_ROOT = Path(CONFIG["output_root"])
STRUCTURED_ROOT = OUTPUT_ROOT / "structured_chapters"
IMAGES_ROOT = OUTPUT_ROOT / "images"
for path in [OUTPUT_ROOT, STRUCTURED_ROOT, IMAGES_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("Environment ready. Outputs to:", OUTPUT_ROOT)


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/home/akash/Desktop/PIHUB/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PermissionError: [Errno 13] Permission denied: '/kaggle'

In [ ]:
PDF_RE = re.compile(r"\.pdf$", re.I)
CLASS_RE = re.compile(r"\bclass\s*([1-9]|10)\b", re.I)
SUBJECT_ALIASES = {
    "math": "mathematics", "maths": "mathematics", "mathematics": "mathematics",
    "science": "science", "social science": "social_science", "social_science": "social_science",
    "biology": "biology", "physics": "physics", "chemistry": "chemistry",
    "history": "history", "geography": "geography", "economics": "economics", "civics": "civics"
}

def norm_subject(value: str) -> str:
    key = str(value or "").lower().replace("-", "_").replace(" ", "_").strip("_")
    return SUBJECT_ALIASES.get(key, key)

def candidate_roots(root: str) -> List[Path]:
    base = Path(root)
    roots = []
    if base.exists():
        roots.append(base)
        roots.extend(sorted([p for p in base.rglob("TEXTBOOKS") if p.is_dir()], key=lambda p: str(p).lower()))
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists() and kaggle_input not in roots:
        roots.append(kaggle_input)
        roots.extend(sorted([p for p in kaggle_input.rglob("TEXTBOOKS") if p.is_dir()], key=lambda p: str(p).lower()))
    return sorted(set(roots), key=lambda p: ("TEXTBOOKS" not in p.parts, len(p.parts)))

def infer_grade(path: Path) -> str:
    for part in path.parts:
        m = CLASS_RE.search(part)
        if m:
            return m.group(1)
    return ""

def infer_subject(path: Path) -> str:
    parts = [p.lower().replace("_", " ") for p in path.parts]
    for raw in SUBJECT_ALIASES.keys():
        normalized = raw.replace("_", " ")
        if normalized in parts:
            return norm_subject(raw)
    try:
        idx = [p.lower() for p in path.parts].index("textbooks")
        if idx + 1 < len(path.parts):
            return norm_subject(path.parts[idx + 1])
    except ValueError:
        pass
    return norm_subject(path.parent.name)

def passes_filters(path: Path) -> bool:
    include_subjects = {norm_subject(s) for s in CONFIG.get("include_subjects", []) if str(s).strip()}
    include_grades = {str(g) for g in CONFIG.get("include_grades", []) if str(g).strip()}
    name_filter = str(CONFIG.get("include_pdf_name_contains") or "").lower().strip()
    if include_subjects and infer_subject(path) not in include_subjects:
        return False
    if include_grades and infer_grade(path) not in include_grades:
        return False
    if name_filter and name_filter not in path.name.lower():
        return False
    return True

def discover_pdfs(root: str) -> List[Path]:
    all_files = []
    for base in candidate_roots(root):
        files = sorted([p for p in base.rglob("*") if p.is_file() and PDF_RE.search(p.name)], key=lambda p: str(p).lower())
        all_files.extend(files)
    unique = sorted(set(all_files), key=lambda p: str(p).lower())
    filtered = [p for p in unique if passes_filters(p)]
    if CONFIG.get("max_pdfs"):
        filtered = filtered[: int(CONFIG["max_pdfs"])]
    return filtered

PDF_FILES = discover_pdfs(CONFIG["pdf_dataset_root"])
inventory = {"pdf_count": len(PDF_FILES)}
(OUTPUT_ROOT / "PDF_DATASET_INVENTORY.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")
print(f"Discovered {len(PDF_FILES)} PDFs")


In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
FORMULA_RE = re.compile(r"([A-Za-z][A-Za-z0-9_ ()]{0,40}\s*(?:=|∝|≤|≥|<|>)\s*[^.;\n]{1,120}|[A-Z][a-z]?\s*=\s*[^.;\n]{1,120})")

def slugify(value: str) -> str:
    text = str(value or "").lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_") or "untitled"

def word_count(text: str) -> int:
    return len(WORD_RE.findall(text or ""))

def make_docling_converter() -> DocumentConverter:
    options = PdfPipelineOptions()
    options.do_ocr = bool(CONFIG["docling_do_ocr"])
    options.do_table_structure = bool(CONFIG["docling_do_table_structure"])
    options.generate_picture_images = bool(CONFIG["docling_generate_picture_images"])
    options.generate_page_images = bool(CONFIG["docling_generate_page_images"])
    options.do_picture_classification = False
    options.do_picture_description = False
    options.do_formula_enrichment = False
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)})

DOCLING_CONVERTER = make_docling_converter()

def pdf_output_path(pdf_path: Path) -> Path:
    grade = f"grade_{infer_grade(pdf_path) or 'unknown'}"
    subject = slugify(infer_subject(pdf_path) or "unknown")
    chapter = slugify(pdf_path.stem)
    return STRUCTURED_ROOT / grade / subject / chapter / "structured_chapter.json"

def convert_pdf_to_structured_chapter(pdf_path: Path):
    out_path = pdf_output_path(pdf_path)
    if CONFIG["resume"] and out_path.exists():
        return json.loads(out_path.read_text(encoding="utf-8"))

    print(f"Converting {pdf_path.name}...")
    result = DOCLING_CONVERTER.convert(str(pdf_path))
    doc_obj = result.document
    
    # Extract Images from doc_obj before dumping to dict
    grade = f"grade_{infer_grade(pdf_path) or 'unknown'}"
    subject = slugify(infer_subject(pdf_path) or "unknown")
    chapter_slug = slugify(pdf_path.stem)
    img_dir = IMAGES_ROOT / grade / subject / chapter_slug
    img_dir.mkdir(parents=True, exist_ok=True)
    
    saved_images = {}
    if hasattr(doc_obj, 'pictures'):
        for idx, pic in enumerate(doc_obj.pictures, start=1):
            fig_id = f"figure_{idx}"
            try:
                pil_image = None
                if hasattr(pic, 'get_image'):
                    image_ref = pic.get_image(doc_obj)
                    if hasattr(image_ref, 'pil_image'):
                        pil_image = image_ref.pil_image
                elif hasattr(pic, 'image') and hasattr(pic.image, 'pil_image'):
                    pil_image = pic.image.pil_image
                if pil_image:
                    img_path = img_dir / f"{fig_id}.png"
                    pil_image.save(img_path, "PNG")
                    saved_images[fig_id] = str(img_path)
            except Exception as e:
                print(f"Warning: Failed to save image {fig_id}: {e}")

    doc = doc_obj.export_to_dict() if hasattr(doc_obj, 'export_to_dict') else doc_obj.model_dump(mode='json')
    
    texts = doc.get("texts") if isinstance(doc.get("texts"), list) else []
    tables = doc.get("tables") if isinstance(doc.get("tables"), list) else []
    pictures = doc.get("pictures") if isinstance(doc.get("pictures"), list) else []

    blocks = []
    for item in texts:
        if not isinstance(item, dict): continue
        text = ""
        for key in ["text", "orig", "content"]:
            if isinstance(item.get(key), str) and item.get(key).strip():
                text = re.sub(r"\s+", " ", item.get(key)).strip()
                break
        if not text: continue
        label = str(item.get("label") or item.get("type") or "").lower()
        kind = "heading" if "title" in label or "section_header" in label else "paragraph"
        equations = [m.group(0).strip() for m in FORMULA_RE.finditer(text)]
        blocks.append({"type": "formula" if equations else kind, "text": text, "equations": equations, "label": label})

    table_objs = []
    for idx, table in enumerate(tables, start=1):
        if isinstance(table, dict):
            data = table.get("data") if isinstance(table.get("data"), dict) else table
            cells = data.get("table_cells") or data.get("cells") or []
            by_row = {}
            for cell in cells:
                if not isinstance(cell, dict): continue
                r, c = int(cell.get("start_row_offset_idx", 0)), int(cell.get("start_col_offset_idx", 0))
                by_row.setdefault(r, []).append((c, str(cell.get("text", "")).strip()))
            rows = [[txt for _, txt in sorted(v)] for _, v in sorted(by_row.items())]
            if rows:
                table_objs.append({"title": str(table.get("caption") or f"Table {idx}"), "headers": rows[0], "rows": rows[1:]})

    figure_objs = []
    for idx, picture in enumerate(pictures, start=1):
        fig_id = f"figure_{idx}"
        caption = str(picture.get("caption") or picture.get("text") or "") if isinstance(picture, dict) else ""
        figure_objs.append({"figure_id": fig_id, "caption": caption, "image_path": saved_images.get(fig_id, "")})

    # Build sections
    sections = []
    current_title = pdf_path.stem
    current_parts, current_equations = [], []
    
    def flush():
        if not current_parts: return
        content = "\n\n".join(current_parts).strip()
        section_id = hashlib.sha256((current_title + content).encode("utf-8")).hexdigest()[:16]
        sections.append({
            "section_id": f"section_{section_id}",
            "title": current_title,
            "level": 1,
            "content": content,
            "tables": [], "figures": [],
            "equations": sorted(set(current_equations)),
            "metadata": {"word_count": word_count(content), "character_count": len(content)}
        })

    for block in blocks:
        text = block.get("text", "")
        if block.get("type") == "heading" and current_parts and word_count("\n".join(current_parts)) >= int(CONFIG["min_section_words"]):
            flush(); current_parts, current_equations = [], []; current_title = text[:120]
        else:
            current_parts.append(text)
            current_equations.extend(block.get("equations", []))
            if word_count("\n".join(current_parts)) >= int(CONFIG["max_section_words"]):
                flush(); current_parts, current_equations = [], []
    flush()
    if sections:
        sections[0]["tables"] = table_objs
        sections[0]["figures"] = figure_objs

    chapter = {
        "document_id": slugify(f"grade_{infer_grade(pdf_path)}_{infer_subject(pdf_path)}_{pdf_path.stem}"),
        "chapter_title": pdf_path.stem,
        "grade": infer_grade(pdf_path),
        "subject": infer_subject(pdf_path),
        "source_pdf": str(pdf_path),
        "sections": sections,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(chapter, indent=2, ensure_ascii=False), encoding="utf-8")
    return chapter

STRUCTURED_CHAPTERS = []
for pdf_path in tqdm(PDF_FILES, desc="Extracting PDFs"):
    STRUCTURED_CHAPTERS.append(convert_pdf_to_structured_chapter(pdf_path))

(OUTPUT_ROOT / "DOCLING_EXTRACTION_MANIFEST.json").write_text(json.dumps({"chapters": len(STRUCTURED_CHAPTERS)}, indent=2), encoding="utf-8")
print("Extraction complete.")
